<details>
   <summary><b>Table of Contents</b></summary>

   - [1. Review Data](#1-review-data)
   - [2. Train - Test Split](#2-train---test-split)
   - [3. Model Selection](#3-model-selection)
   - [4. Test with Best Model](#4-test-with-best-model)
   - [5. Tuning Model](#5-tuning-model)
   - [6. Evaluate Model](#6-evaluate-model)
     - [6.1. Train - Test Split](#61-train---test-split)
     - [6.2. Cross Validation (K-Fold)](#62-cross-validation-k-fold)
     - [6.3. Visualization](#63-visualization)
   - [7. Save Model](#7-save-model)
</details>

In [1]:
import warnings
import pandas as pd

from sklearn.pipeline      import Pipeline
from sklearn.compose       import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.svm          import SVR
from sklearn.tree         import DecisionTreeRegressor
from sklearn.neighbors    import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble     import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost              import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics         import root_mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

pd.set_option('display.float_format', lambda x: f'{x:.3e}')

## **1. Review Data**
---

In [2]:
df = pd.read_csv('../data/data_model.csv')
df.head()

,Company Name,Location,Headquarters,Size,Type of ownership,Industry,Sector,Revenue,job_simplified,seniority,Rating Category,job_state,Python_yn,Spark,AWS_yn,Average Salary
0,Tecolote Research,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,Company - Private,Aerospace & Defense,Aerospace & Defense,$50 to $100 million (USD),data scientist,Other,Medium Rating,NM,1,0,0,7.200e+04
1,University of Maryland Medical System,"Linthicum, MD","Baltimore, MD",10000+ employees,Other Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),data scientist,Other,Medium Rating,MD,1,0,0,8.750e+04
2,KnowBe4,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,Company - Private,Security Services,Business Services,$100 to $500 million (USD),data scientist,Other,High Rating,FL,1,1,0,8.500e+04
3,PNNL,"Richland, WA","Richland, WA",1001 to 5000 employees,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),data scientist,Other,Medium Rating,WA,1,0,0,7.650e+04
4,Affinity Solutions,"New York, NY","New York, NY",51 to 200 employees,Company - Private,Advertising & Marketing,Business Services,Unknown,data scientist,Other,Low Rating,NY,1,0,0,1.145e+05


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 742 entries, 0 to 741
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Company Name       742 non-null    object 
 1   Location           742 non-null    object 
 2   Headquarters       742 non-null    object 
 3   Size               742 non-null    object 
 4   Type of ownership  742 non-null    object 
 5   Industry           742 non-null    object 
 6   Sector             742 non-null    object 
 7   Revenue            742 non-null    object 
 8   job_simplified     742 non-null    object 
 9   seniority          742 non-null    object 
 10  Rating Category    742 non-null    object 
 11  job_state          742 non-null    object 
 12  Python_yn          742 non-null    int64  
 13  Spark              742 non-null    int64  
 14  AWS_yn             742 non-null    int64  
 15  Average Salary     742 non-null    float64
dtypes: float64(1), int64(3), o

In [4]:
df.describe()

,Python_yn,Spark,AWS_yn,Average Salary
count,7.420e+02,7.420e+02,7.420e+02,7.420e+02
mean,5.283e-01,2.251e-01,2.372e-01,1.015e+05
std,4.995e-01,4.179e-01,4.257e-01,3.746e+04
min,0.000e+00,0.000e+00,0.000e+00,1.550e+04
25%,0.000e+00,0.000e+00,0.000e+00,7.350e+04
50%,1.000e+00,0.000e+00,0.000e+00,9.750e+04
75%,1.000e+00,0.000e+00,0.000e+00,1.225e+05
max,1.000e+00,1.000e+00,1.000e+00,2.540e+05


## **2. Train - Test Split**
---

In [5]:
X = df.drop('Average Salary', axis=1)
y = df['Average Salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"y_test  shape: {y_test.shape}")

X_train shape: (593, 15)
y_train shape: (593,)
X_test  shape: (149, 15)
y_test  shape: (149,)


## **3. Model Selection**
---

In [6]:
def create_regression_pipeline(model_name):
    regression_models = {
        'SVR'                      : SVR(),
        'XGBoost'                  : XGBRegressor(random_state=42),
        'KNeighbors'               : KNeighborsRegressor(),
        'DecisionTree'             : DecisionTreeRegressor(random_state=42),
        'RandomForest'             : RandomForestRegressor(random_state=42),
        'LinearRegression'         : LinearRegression(),
        'HistogramGradientBoosting': HistGradientBoostingRegressor(random_state=42),
    }

    categorical_features = X.select_dtypes(include=['object']).columns.tolist()
    numerical_features   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
        ]
    )

    regression_pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('regressor'   , regression_models[model_name])
        ]
    )

    pipeline_dict = {model_name: regression_pipeline}
    return pipeline_dict


def create_all_regression_pipelines():
    regression_model_names = ['RandomForest', 'SVR', 'LinearRegression', 'KNeighbors', 'DecisionTree', 'XGBoost', 'HistogramGradientBoosting']

    all_pipelines = {}
    for model_name in regression_model_names:
        pipeline_dict = create_regression_pipeline(model_name)
        all_pipelines.update(pipeline_dict)
        
    return all_pipelines


def train_evaluate_model_with_df(model_name, model, X_train, y_train, X_test, y_test, results_df):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    model_results = pd.DataFrame({'Model': [model_name], 'RMSE': rmse, 'MAE': mae, 'R2': r2})
    results_df    = pd.concat([results_df, model_results], ignore_index=True)
    return results_df


def run_pipelines_with_metrics_to_dataframe(all_pipelines, X_train, y_train, X_test, y_test):
    results_df = pd.DataFrame(columns=['Model', 'RMSE', 'MAE', 'R2'])

    for model_name, pipeline in all_pipelines.items():
        results_df = train_evaluate_model_with_df(
            model_name, pipeline, 
            X_train, y_train, X_test, y_test, 
            results_df
        )

    results_df = results_df.sort_values(by='R2', ascending=False).reset_index(drop=True)
    return results_df

In [7]:
all_pipelines = create_all_regression_pipelines()
results_df    = run_pipelines_with_metrics_to_dataframe(all_pipelines, X_train, y_train, X_test, y_test)
results_df

,Model,RMSE,MAE,R2
0,XGBoost,1.702e+04,1.056e+04,8.153e-01
1,RandomForest,1.714e+04,1.127e+04,8.127e-01
2,HistogramGradientBoosting,2.002e+04,1.456e+04,7.443e-01
3,DecisionTree,2.146e+04,1.024e+04,7.062e-01
4,LinearRegression,2.150e+04,1.212e+04,7.050e-01
5,KNeighbors,2.834e+04,2.114e+04,4.878e-01
6,SVR,4.020e+04,3.117e+04,-3.078e-02


> **Observations**:  
>  
> Best-performing models:  
>  
> - **XGBoost** has the lowest RMSE (1.702e+04) and MAE (1.056e+04) with the highest R² (0.8153), making it the top performer.  
> - **Random Forest** follows closely with similar RMSE and MAE but slightly lower R² (0.8127).  
>  
> Moderate performance:  
>  
> - **Histogram Gradient Boosting**, **Decision Tree**, and **Linear Regression** show decent results but with higher errors and lower R² scores (~0.70).  
>  
> Poor performance:  
>  
> - **K-Neighbors** has significantly higher RMSE (2.834e+04) and MAE (2.114e+04), leading to a poor R² (0.4878).  
> - **SVR** performs worst, with extremely high RMSE (4.020e+04) and MAE (3.117e+04), and a negative R² (-0.0378), indicating it fails to explain variance.  

> **Conclusion**:  
>  
> - **XGBoost** is the best model, offering the best balance of error minimization and predictive power.  
> - **Random Forest** is a strong alternative, with only a slight drop in performance.  
> - **SVR** and **K-Neighbors** are ineffective for this dataset and should be avoided.  

## **4. Test with Best Model**
---

In [8]:
xgb_res        = XGBRegressor()
default_params = xgb_res.get_params()

desired_params          = ['reg_lambda', 'reg_alpha', 'n_estimators', 'max_depth', 'learning_rate', 'gamma', 'colsample_bytree']
selected_default_params = {param: default_params[param] for param in desired_params}

default_params_df = pd.DataFrame([selected_default_params])
default_params_df = default_params_df.transpose()
default_params_df.columns = ['Value']

print('XGBRegressor default params:')
default_params_df.T

XGBRegressor default params:


,reg_lambda,reg_alpha,n_estimators,max_depth,learning_rate,gamma,colsample_bytree
Value,None,None,None,None,None,None,None


In [9]:
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

xgb_res = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', XGBRegressor())
    ]
)

xgb_res.fit(X_train, y_train)
y_pred = xgb_res.predict(X_test)

print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.3f}")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.3f}")
print(f"R2  : {r2_score(y_test, y_pred):.3f}")

RMSE: 17018.299
MAE : 10560.083
R2  : 0.815


## **5. Implementing Optuna for Hyperparameter Tuning**
---

In [10]:
import optuna
from optuna.pruners          import MedianPruner
from sklearn.base            import clone
from sklearn.model_selection import KFold, train_test_split
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import OneHotEncoder, StandardScaler
from xgboost                 import XGBRegressor
from sklearn.metrics         import root_mean_squared_error, mean_absolute_error, r2_score

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [11]:
# Read data
df = pd.read_csv('../data/data_model.csv')
X  = df.drop('Average Salary', axis=1)
y  = df['Average Salary']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ]
)

In [12]:
def objective(trial: optuna.Trial):
    """Define the objective function for Optuna optimization."""

    # Hyperparameter space
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 1500, step=50),
        'learning_rate'   : trial.suggest_float('learning_rate', 1e-3, 1, log=True),
        'max_depth'       : trial.suggest_int('max_depth', 3, 21),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma'           : trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        # For faster training
        'tree_method' : 'hist',
        'random_state': 42,
        'n_jobs'      : -1,
    }

    # KFold cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []

    # Cross-validation loop and early stopping (each fold will have its own eval_set)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), start=1):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        prep      = clone(preprocessor)
        X_tr_enc  = prep.fit_transform(X_tr)
        X_val_enc = prep.transform(X_val)

        reg = XGBRegressor(**params, early_stopping_rounds=50)
        reg.fit(
            X_tr_enc, y_tr,
            eval_set = [(X_val_enc, y_val)],
            verbose  = False
        )

        y_pred = reg.predict(X_val_enc)
        rmse   = root_mean_squared_error(y_val, y_pred)
        rmse_scores.append(rmse)

        # Report intermediate results to Optuna and handle pruning
        trial.report(rmse, step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # Object: minimize mean RMSE
    return float(np.mean(rmse_scores))

In [13]:
# Create Optuna study
study = optuna.create_study(
    direction = 'minimize',
    sampler   = optuna.samplers.TPESampler(seed=42),
    pruner    = MedianPruner(n_warmup_steps=5),
)
study.optimize(objective, n_trials=1000, show_progress_bar=True, n_jobs=-1)

[I 2025-08-27 13:09:43,355] A new study created in memory with name: no-name-29d692da-d0c7-41f3-9218-9d3262ae395b


  0%|          | 0/1000 [00:00<?, ?it/s]

[I 2025-08-27 13:09:51,742] Trial 4 finished with value: 20831.948925062545 and parameters: {'n_estimators': 250, 'learning_rate': 0.35774296537689615, 'max_depth': 9, 'min_child_weight': 2.6510557815541125, 'subsample': 0.6392187271053276, 'colsample_bytree': 0.5247291485385017, 'gamma': 4.149450398523152, 'reg_alpha': 5.523729242993812e-06, 'reg_lambda': 8.417578635464899e-08}. Best is trial 4 with value: 20831.948925062545.
[I 2025-08-27 13:09:54,198] Trial 9 finished with value: 21745.42088840127 and parameters: {'n_estimators': 700, 'learning_rate': 0.24272658107229353, 'max_depth': 21, 'min_child_weight': 2.308840271418345, 'subsample': 0.5540474144374554, 'colsample_bytree': 0.8362821508743072, 'gamma': 3.83452661308425, 'reg_alpha': 2.2149553025563018e-05, 'reg_lambda': 0.0019376760139358826}. Best is trial 4 with value: 20831.948925062545.
[I 2025-08-27 13:09:56,032] Trial 7 finished with value: 22042.80661532733 and parameters: {'n_estimators': 550, 'learning_rate': 0.4110551

In [14]:
# Report the best trial
print('Best trial :', study.best_trial.number)
print('Best RMSE  :', study.best_value)
print('Best params:')
for k, v in study.best_params.items():
    print(f"\t{k}: {v}")

Best trial : 775
Best RMSE  : 19048.01270075514
Best params:
	n_estimators: 1000
	learning_rate: 0.2259819565429738
	max_depth: 11
	min_child_weight: 1.477327898415538
	subsample: 0.8990850732677914
	colsample_bytree: 0.5103249628264016
	gamma: 2.792911522391636
	reg_alpha: 2.501904739969501e-07
	reg_lambda: 0.18999603922081723


In [15]:
# Re-train the model with the best hyperparameters
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

prep      = clone(preprocessor)
X_tr_enc  = prep.fit_transform(X_tr)
X_val_enc = prep.transform(X_val)

best_reg = XGBRegressor(
    **study.best_params,
    tree_method           = 'hist',
    random_state          = 42,
    n_jobs                = -1,
    early_stopping_rounds = 50,
)

best_reg.fit(
    X_tr_enc, y_tr,
    eval_set = [(X_val_enc, y_val)],
    verbose  = False,
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.5103249628264016
,device,None
,early_stopping_rounds,50
,enable_categorical,False
,eval_metric,None


## **6. Evaluation**
---

In [22]:
from sklearn.pipeline import Pipeline
best_model = Pipeline([
    ('preprocessor', prep),
    ('regressor'   , best_reg),
])

y_pred = best_model.predict(X_test)

print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.3f}")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.3f}")
print(f"R2  : {r2_score(y_test, y_pred):.3f}")

RMSE: 17954.026
MAE : 9503.655
R2  : 0.794


## **7. Save Model**
---

In [23]:
import os
import joblib

OUTPUT_DIR  = '../outputs/optuna_xgboost'
os.makedirs(OUTPUT_DIR, exist_ok=True)

joblib.dump(best_model, f"{OUTPUT_DIR}/xgb_optuna_tuning.pkl")
print(f'XGBoost model with Optuna tuning saved to {OUTPUT_DIR}/xgb_optuna_tuning.pkl')

XGBoost model with Optuna tuning saved to ../outputs/optuna_xgboost/xgb_optuna_tuning.pkl
